In [2]:
import pandas as pd
from sqlalchemy import text
%run db_config.ipynb

def load_symbol_id_map(conn) -> dict[str, int]:
    rows = conn.execute(text("SELECT id, symbol FROM stocks")).fetchall()
    return {row.symbol: row.id for row in rows}

import yfinance as yf

def download_ohlc(symbol: str, period: str = "10y") -> pd.DataFrame:
    # yfinance uses hyphen for class shares: BRK.B -> BRK-B
    yf_symbol = symbol.replace(".", "-")
    df = yf.download(yf_symbol, period=period, interval="1d", auto_adjust=False)
    if df.empty:
        return df

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df = df.rename(columns={
        "Date": "trade_date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df = df[["trade_date", "open", "high", "low", "close", "volume"]]
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date
    df = df.dropna(subset=["open", "high", "low", "close", "volume"])
    return df

def insert_ohlc_for_symbol(conn, stock_id: int, df: pd.DataFrame, chunk_size: int = 500) -> int:
    df = df.copy()
    df["stock_id"] = stock_id
    inserted = 0

    for start in range(0, len(df), chunk_size):
        chunk = df.iloc[start : start + chunk_size]

        # Skip rows where trade_date already exists for this stock
        dates = chunk["trade_date"].tolist()
        existing = conn.execute(
            text(
                "SELECT trade_date FROM stock_ohlc_data "
                "WHERE stock_id = :stock_id AND trade_date = ANY(:dates)"
            ),
            {"stock_id": stock_id, "dates": dates},
        ).fetchall()
        existing_dates = {row.trade_date for row in existing}
        chunk = chunk[~chunk["trade_date"].isin(existing_dates)]

        if chunk.empty:
            continue

        records = chunk.to_dict(orient="records")
        conn.execute(
            text(
                "INSERT INTO stock_ohlc_data "
                "(trade_date, open, high, low, close, volume, stock_id) "
                "VALUES (:trade_date, :open, :high, :low, :close, :volume, :stock_id)"
            ),
            records,
        )
        inserted += len(records)

    return inserted

import json
import time
from pathlib import Path

CHECKPOINT_FILE = Path("python/checkpoint/ohlc_checkpoint.json")

def load_checkpoint() -> set[str]:
    if CHECKPOINT_FILE.exists():
        return set(json.loads(CHECKPOINT_FILE.read_text()))
    return set()

def save_checkpoint(done: set[str]) -> None:
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(sorted(done)))

def main():
    engine = get_engine()
    done = load_checkpoint()

    with engine.begin() as conn:
        symbol_map = load_symbol_id_map(conn)

    for symbol, stock_id in symbol_map.items():
        if symbol in done:
            print(f"SKIP {symbol} (checkpoint)")
            continue

        try:
            df = download_ohlc(symbol)
            if df.empty:
                print(f"WARN {symbol}: no data")
                done.add(symbol)
                save_checkpoint(done)
                continue

            with engine.begin() as conn:
                count = insert_ohlc_for_symbol(conn, stock_id, df)

            print(f"OK {symbol}: inserted {count} rows")
            done.add(symbol)
            save_checkpoint(done)
            time.sleep(0.5)  # light throttle for yfinance

        except Exception as e:
            print(f"FAIL {symbol}: {e}")

    print(f"\nCompleted {len(done)} / {len(symbol_map)} symbols")

if __name__ == "__main__":
    main()

[*********************100%***********************]  1 of 1 completed


OK MMM: inserted 2514 rows


[*********************100%***********************]  1 of 1 completed


OK AOS: inserted 2514 rows


[*********************100%***********************]  1 of 1 completed


...

OK ZBRA: inserted 2514 rows


[*********************100%***********************]  1 of 1 completed


OK ZBH: inserted 2514 rows


[*********************100%***********************]  1 of 1 completed


OK ZTS: inserted 2514 rows

Completed 503 / 503 symbols
